# TRÍCH XUẤT ĐẶC TRƯNG ĐA PHƯƠNG THỨC (FEATURE ENGINEERING & MULTIMODAL INTEGRATION)

Notebook này thực hiện trích xuất thuộc tính cấu trúc ($X_s$), thuộc tính văn bản ($X_t$) và tích hợp thành ma trận đa phương thức ($X_c$), đồng thời thực hiện phân chia dữ liệu Out-of-Time để phục vụ cho các mô hình học máy.

## Cell 1: Khai báo thư viện & Thiết lập môi trường

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs('data/processed', exist_ok=True)
os.makedirs('report', exist_ok=True)

print("✓ Khai báo thư viện và môi trường thành công.")

## Cell 2: Tải dữ liệu từ DuckDB / SQL Feature Table

In [ ]:
con = duckdb.connect('data/processed/ady.duckdb')
feat = con.execute("SELECT * FROM feat").df()

print(f"✓ Tổng số bản ghi: {len(feat):,} dòng, {feat.shape[1]} cột")
print("\nPhân bố các nhãn mục tiêu (y):")
print(feat['y'].value_counts(normalize=True) * 100)

## Cell 3: Phân chia dữ liệu theo chuỗi thời gian (Out-of-Time Train/Test Split)
> **Lưu ý quan trọng:** Để tránh rò rỉ dữ liệu (Data Leakage), phân chia dữ liệu nghiêm ngặt theo mốc thời gian `year`: Tập **Train** gồm các sự cố từ năm 2023 trở về trước, tập **Test** gồm các sự cố từ năm 2024 trở đi.

In [ ]:
train = feat[feat['year'] <= 2023].copy()
test  = feat[feat['year'] >= 2024].copy()

print(f"✓ Tập Train (<= 2023): {len(train):,} bản ghi ({len(train)/len(feat):.1%})")
print(f"✓ Tập Test  (>= 2024): {len(test):,} bản ghi ({len(test)/len(feat):.1%})")

assert len(test) >= 0.10 * len(train), "Tập Test quá nhỏ, cần điều chỉnh mốc năm!"

## Cell 4: Tiền xử lý & Trích xuất thuộc tính cấu trúc (Structured Features - $X_s$)

In [ ]:
top_aircraft = train['aircraft_type'].value_counts().head(10).index
train['aircraft_type'] = train['aircraft_type'].apply(lambda x: x if x in top_aircraft else 'Other')
test['aircraft_type']  = test['aircraft_type'].apply(lambda x: x if x in top_aircraft else 'Other')

alt_median = train['altitude'].median()
train['altitude'] = train['altitude'].fillna(alt_median)
test['altitude']  = test['altitude'].fillna(alt_median)

cat_cols = ['flight_phase', 'weather', 'light', 'airspace', 'mission', 'aircraft_type']
Xs_train = pd.get_dummies(train[cat_cols + ['altitude', 'is_uas']], columns=cat_cols, drop_first=True)
Xs_test  = pd.get_dummies(test[cat_cols + ['altitude', 'is_uas']], columns=cat_cols, drop_first=True)

Xs_train, Xs_test = Xs_train.align(Xs_test, join='left', axis=1, fill_value=0)

print(f"✓ Số lượng thuộc tính cấu trúc (Xs): {Xs_train.shape[1]} cột")

## Cell 5: Trích xuất thuộc tính văn bản bằng TF-IDF (Text Features - $X_t$)

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True,
    stop_words='english'
)
Xt_train = tfidf.fit_transform(train['narrative'].fillna(''))
Xt_test  = tfidf.transform(test['narrative'].fillna(''))

print(f"✓ Ma trận TF-IDF Train (Xt): {Xt_train.shape}")
print(f"✓ Ma trận TF-IDF Test  (Xt): {Xt_test.shape}")

## Cell 6: Tích hợp đa phương thức (Multimodal Combined Features - $X_c$)

In [ ]:
Xs_train_sparse = csr_matrix(Xs_train.values.astype(np.float32))
Xs_test_sparse  = csr_matrix(Xs_test.values.astype(np.float32))

Xc_train = hstack([Xs_train_sparse, Xt_train]).tocsr()
Xc_test  = hstack([Xs_test_sparse, Xt_test]).tocsr()

print(f"✓ Ma trận Đa phương thức Train (Xc): {Xc_train.shape}")
print(f"✓ Ma trận Đa phương thức Test  (Xc): {Xc_test.shape}")

## Cell 7: Tính toán ngưỡng đánh giá cơ sở (Naïve Baseline Metric)

In [ ]:
majority_class = train['y'].mode()[0]
naive_preds = np.full(shape=len(test), fill_value=majority_class)

naive_f1 = f1_score(test['y'], naive_preds, average='macro')

print(f"=== NAÏVE BASELINE RESULTS ===")
print(f"Dự đoán nhãn đa số: '{majority_class}'")
print(f"Naïve Baseline Macro-F1 Score: {naive_f1:.4f}")
print("\nBáo cáo chi tiết:")
print(classification_report(test['y'], naive_preds, zero_division=0))

## Cell 8: Lưu trữ kết quả & Báo cáo thuộc tính

In [ ]:
feat_summary = pd.DataFrame({
    'feature_name': list(Xs_train.columns) + [f"tfidf_word_{i}" for i in range(Xt_train.shape[1])],
    'modality': ['Structured'] * Xs_train.shape[1] + ['Text_TFIDF'] * Xt_train.shape[1]
})
feat_summary.to_csv('report/table_features.csv', index=False)

save_npz('data/processed/Xc_train.npz', Xc_train)
save_npz('data/processed/Xc_test.npz', Xc_test)

np.save('data/processed/y_train.npy', train['y'].values)
np.save('data/processed/y_test.npy', test['y'].values)

print("✓ Đã xuất thành công: report/table_features.csv và các file ma trận trong data/processed/")

## Cell 9: Nhật ký AI Audit Log (Minh chứng Human Delta)

### AI Audit Log - Step 4: Feature Engineering
* **Prompt được sử dụng:** "Tạo mã nguồn trích xuất đặc trưng đa phương thức (TF-IDF + Cấu trúc) và chia dữ liệu theo thời gian."
* **Sự cố / Lỗi phát hiện được:** AI ban đầu sử dụng `train_test_split(random_state=42)` ngẫu nhiên, dẫn đến rò rỉ dữ liệu chuỗi thời gian (Temporal Leakage).
* **Human Delta (Can thiệp của sinh viên):**
  1. Thay thế bằng phân tách Out-of-Time dựa trên thuộc tính `year` (Train: <=2023, Test: >=2024).
  2. Đảm bảo hàm `tfidf.fit_transform()` chỉ được gọi trên tập Train, tập Test chỉ dùng `tfidf.transform()`.